In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
from minsearch import Index


documents = []



for file in files:
    doc = file.parse()
    documents.append(doc)
    
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)
index.fit(documents)

print(len(documents))

72


In [4]:

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [5]:
question = ("How does the agentic loop keep calling the model until it stops?")
results = index.search(question)

In [6]:
print(results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


In [17]:
INSTRUCTIONS = '''
You are an assistant that answers questions strictly based on the provided context.

The context consists of multiple documents, each with:

filename: the name of the file

content: the text extracted from that file

Your job is to:

Read the question

Use the retrieved context chunks

Produce an accurate answer using ONLY the information in the context

Rules:

If the answer is found in the context, answer clearly and concisely.

If the answer is not found in the context, respond with: "I don't know."

Do NOT invent facts.

Do NOT use outside knowledge.

Your output must be only the final answer.
'''
USER_PROMPT_TEMPALATE = '''
Question:
{question}

Context:
{context}
'''

In [18]:
def search(question):
    boost_dict = {"content": 2.0, "filename": 0.5}


    return index.search(
        question,
        boost_dict=boost_dict,
        num_results=5)


search_result = search(question)

In [19]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['filename'])
        lines.append('C: ' + doc['content'])
        lines.append('')

    return '\n'.join(lines).strip()

In [20]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [21]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    print (response.usage)

    return response.output_text

In [22]:
question = "How does the agentic loop keep calling the model until it stops?"

def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [28]:
answer = rag(question)
print(answer)

ResponseUsage(input_tokens=7197, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=65, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7262)
The loop keeps calling the model by checking whether the response contains any function calls. If it does, the code runs the tool, appends the tool result to the message history, and calls the model again. It stops when a turn returns no function calls, meaning the model has given a final answer.


In [25]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))


295


In [ ]:
chunks_documents = []
for chunk in chunks:
    doc = {
        'filename': chunk['filename'],
        'content': chunk['content']
    }
    chunks_documents.append(doc)

In [ ]:
chunks_result = search(question)


In [ ]:
def rag_chunks(query, model='gpt-5.4-mini'):
    chunks_result = search(question)
    prompt = build_prompt(query, chunks_result)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [29]:
answer = rag_chunks(question)
print(answer)

ResponseUsage(input_tokens=7197, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=76, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7273)
It keeps calling the model inside a `while True` loop, checking each response for function calls. If the model returns any `function_call`, the code runs the tool, appends the tool result to the message history, and loops again. It stops when a response has no function calls, using `has_function_calls == False` as the exit condition.


In [30]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the loaded chunks for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'content': 3.0, 'filename': 0.5},
    )

In [32]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [33]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the loaded files for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [34]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [35]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [36]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received
